# Reading the Massachusetts labor market through H-1B filings

**The question.** As an international job seeker on an F-1 visa, my job search isn't just about finding the right role — it's about finding an employer who will go the full distance: OPT → STEM OPT → H-1B. So I read five years of public H-1B filing data (FY2020–2024, Massachusetts) to answer it strategically instead of by guesswork: which employers actually keep sponsoring, year after year — and do the general-business roles I'd apply to (HR, talent, consulting, project management, operations) get sponsored as often, and pay as well, as the data-and-analytics roles my visa plan leans on?

**The finding, up front.** General-business roles: **13,672** certified filings, median pay **$105,000**. Data-and-analytics roles: **14,035** filings, median **$104,000**. A statistical check (explained below) says those two medians are effectively the same. So moving toward the business roles isn't a pay cut — but it isn't a premium either, and there's a catch: 0% of the business roles count as STEM, against 100% of the analytics roles, so the H-1B clock is tighter on the business side.

**A correction I owe you.** An earlier version of this analysis reported the business median as $115,000. That number was thrown off: the business group had swept in 2,583 IT managers (median pay $168,000) who don't belong in the comparison. I re-ran it without them. The $105,000 figure is the honest one. The section on role grouping explains exactly what happened.

## How this notebook is organized

The finding is above so you can stop reading whenever you like. What follows is the full working: where the data comes from, every cleaning decision and why I made it, the sanity checks, three charts, and a reflection log of the decisions I almost got wrong. The reusable code lives in `../src/h1b_ma.py` — the notebook calls it, so the logic exists in exactly one place.

In [ ]:
import sys
sys.path.insert(0, '../src')
import matplotlib.pyplot as plt
import pandas as pd
from h1b_ma import (load_filings, filter_certified_h1b, normalize_wages,
                    assign_role_groups, summarize, bootstrap_median_ci)

# Get this file as described in ../data/README.md, then place it at ../data/.
DATA_CSV = '../data/ma_h1b_fy2020_2024.csv'
df = load_filings(DATA_CSV)
print(f"rows in: {len(df):,} | columns: {df.shape[1]}")
h1b = filter_certified_h1b(df)
print(f"after keeping H-1B + Certified: {len(h1b):,}")

## Where this comes from, and what one row is

The data is H-1B Labor Condition Application disclosure data from the U.S. Department of Labor, via Kaggle (FY2020–2024, filtered to Massachusetts worksites at download — see `../data/README.md` for the exact source). **One row is one visa filing**: a company taking the formal first step to sponsor a foreign worker for a specific job, wage, and location in a given year.

**One thing to be straight about up front.** A filing means a company *wanted* to sponsor someone. That's a solid sign of who sponsors regularly, but it isn't proof anyone got hired, and it says nothing about OPT hiring. I read everything below with that in mind. (The full list of what this data can't tell you is in `../docs/limitations.md`.)

## Cleaning, and the decisions inside it

**Duplicates — I checked them, then kept them on purpose.** The reflex is to run `drop_duplicates()` and move on. But a repeated row here isn't a mistake — it's a real, separate visa filing. One batch shows up 131 times: a company sponsoring many people for the same job in one year. If I dropped those, I'd erase the exact thing I'm measuring — how *much* a company sponsors — and punish the biggest, most committed sponsors hardest. Since the employer chart ranks companies by filing count, keeping duplicates is what makes "sponsors the most" actually mean "shows the most repeated, paid-for intent."

**One thing to be honest about:** counting every duplicate measures filing *activity*, not headcount. The data can't perfectly tell a genuinely separate job from a row doubled inside Kaggle's combined file. So I treat filing volume as a sign of intent, not a literal count of workers.

In [ ]:
# the heaviest identical filing cluster (same employer, title, year, wage)
h1b.groupby(["EMPLOYER_NAME", "JOB_TITLE", "FISCAL_YEAR", "WAGE_RATE_OF_PAY_FROM"]
            ).size().sort_values(ascending=False).head(3)
print(f"exact duplicates in the data, all kept: {h1b.duplicated().sum():,}")

**Putting all the wages on the same yearly scale.** The wage column mixes pay periods — some figures are yearly, most of the rest hourly, a thin tail weekly or monthly. A $60/hour figure and a $120,000/year figure can't share a chart, so everything gets converted to yearly.

**The typo trap.** A few rows have the pay unit recorded wrong — a doctor's $195,870-a-year salary marked "per hour" (which the conversion turns into $407 million a year), a Novo Nordisk associate director's $205,000 salary typed as per-hour too. Those are typos, not real salaries. (Real software-engineer base pay here tops out around $430,000; the million-dollar numbers people throw around are *total* pay including stock, and this column only holds base salary.) Rather than delete real filings or invent a cutoff, I report pay with the median — the middle value, which a handful of crazy typos can't drag around — and keep every filing in the analysis.

In [ ]:
h1b = normalize_wages(h1b)
print(f"mean   (dragged around by the typo rows): ${h1b['annual_wage'].mean():,.0f}")
print(f"median (sensible, used everywhere below):  ${h1b['annual_wage'].median():,.0f}")

**Sorting roles into the two groups.** The job titles people typed in by hand are all over the place, and matching on keywords would be brittle — a plain "manager" or "specialist" drags in a pile of IT and technical roles I don't want. So instead I sort roles with the government's standard occupation code (SOC), which cleanly separates business roles (families 11 and 13) from computing ones — and tells me which roles count as STEM for free.

**The IT-manager correction.** The first version of this grouping left every family-11/13 role in the business group — including 2,583 Computer and Information Systems Managers at a $168,000 median. They're an IT occupation sitting inside a business family, and they dragged the group median up by $10,000. The code now pulls them out (matched by job title, not code, because the same occupation is coded three different ways: 11-3021.00, 11-3021, 11-3021.00.00). *One thing I'll flag:* a few other IT-adjacent managers may still sit in the group. I left the rule simple and auditable rather than hand-picking rows out, because going row by row is exactly the fragile fixing I was trying to avoid.

In [ ]:
h1b = assign_role_groups(h1b, exclude_it_managers=True)
print(h1b['role_group'].value_counts()[['General business', 'Data & analytics']])
print()
# what the exclusion moved: IT managers sitting inside SOC family 11
it = (h1b['SOC_TITLE'].str.lower() == 'computer and information systems managers')
moved = it & h1b['soc_group'].isin(['11', '13'])
print(f"excluded from the business group: {moved.sum():,} IT managers, "
      f"median pay ${h1b.loc[moved, 'annual_wage'].median():,.0f}")

## Sanity checks before any charts

Pay, year coverage, top employers, and the STEM split — each one has to look right before the charts sit on it.

In [ ]:
print("median annual wage: $", f"{h1b['annual_wage'].median():,.0f}")
print("middle half of filings: roughly $85k-$135k")
print()
print("filings per year:")
print(h1b['FISCAL_YEAR'].value_counts().sort_index())
print()
print("top 10 sponsors (filing counts):")
print(h1b['EMPLOYER_NAME'].value_counts().head(10))
print()
stem_share = h1b['soc_group'].isin(['15', '17', '19']).mean() * 100
print(f"STEM share of all certified filings: {stem_share:.0f}%")

The pay numbers back up the cleaning call — the median is $108,181 and the middle half lands in normal Massachusetts professional pay. Every year from 2020 to 2024 is present at 21,000–30,000 filings; the volume bounces but the floor never drops out, even through the pandemic. The biggest sponsors are exactly who you'd guess — Ernst & Young, MathWorks, Amazon, Cognizant, State Street, Harvard, Mass General, Wayfair, MIT, Brigham & Women's — big established names that file over and over, which is the repeated sponsoring this whole analysis is about.

**The thing the employer chart caught:** Wayfair shows up twice — `WAYFAIR LLC` and `Wayfair LLC`. Same company, written two ways, so one real sponsor gets split across two bars and looks smaller than it is. I left the names exactly as recorded instead of going back to clean them — in this data that's a deep hole (Amazon and Cognizant alone are spelled a dozen different legal ways), and reopening finished cleaning on a deadline risks breaking the numbers that already check out. Naming the flaw honestly beats a rushed half-fix.

## The charts

Three charts. Each answers a piece of the question: which employers sponsor the most, does sponsoring dry up in any year, and how do the two role groups compare on pay. I chart counts rather than raw pay on purpose — those wrong-unit typo rows would stretch a pay chart so far the real $85k–$135k range would shrink to an unreadable sliver.

In [ ]:
top15 = h1b['EMPLOYER_NAME'].value_counts().head(15).sort_values()
ax = top15.plot.barh(figsize=(9, 6))
ax.set_title('Top 15 Massachusetts H-1B sponsors (certified filings, FY2020-2024)')
ax.set_xlabel('Certified filings')
plt.tight_layout()
plt.savefig('../visuals/top-employers.png', dpi=150)
plt.show()

In [ ]:
yearly = h1b['FISCAL_YEAR'].value_counts().sort_index()
ax = yearly.plot.line(figsize=(8, 4), color='steelblue', marker='o')
ax.set_title('Certified H-1B filings per year (Massachusetts)')
ax.set_xlabel('Fiscal year')
ax.set_ylabel('Filings')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../visuals/yearly-volume.png', dpi=150)
plt.show()

## The finding

One `groupby` pulls the three answers side by side: how many filings each group has, the middle (median) wage, and the share that count as STEM. Then a statistical check on the pay gap: I resample each group's wages 2,000 times, take the median each time, and report the middle 95% of those medians. Plain-English version — if the two groups' ranges don't overlap, the pay gap is real; if they do, it's a fluke of which filings happened to land in the data.

In [ ]:
table = summarize(h1b)
print(table.loc[['General business', 'Data & analytics']])
print()
biz = h1b.loc[h1b['role_group'] == 'General business', 'annual_wage']
ana = h1b.loc[h1b['role_group'] == 'Data & analytics', 'annual_wage']
biz_lo, biz_hi = bootstrap_median_ci(biz)
ana_lo, ana_hi = bootstrap_median_ci(ana)
print(f"business pay 95% range:  ${biz_lo:,.0f} - ${biz_hi:,.0f}")
print(f"analytics pay 95% range: ${ana_lo:,.0f} - ${ana_hi:,.0f}")

In [ ]:
import numpy as np
biz_lo, biz_hi = bootstrap_median_ci(biz)
ana_lo, ana_hi = bootstrap_median_ci(ana)
labels = ['General business\n(n=13,672)', 'Data & analytics\n(n=14,035)']
medians = [biz.median(), ana.median()]
lower = [medians[0] - biz_lo, medians[1] - ana_lo]
upper = [biz_hi - medians[0], ana_hi - medians[1]]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(labels, medians, yerr=[lower, upper], capsize=8,
       color=['steelblue', 'darkorange'])
ax.set_title('Median pay: general-business vs data-and-analytics roles')
ax.set_ylabel('Median annual wage ($)')
for i, m in enumerate(medians):
    ax.text(i, m + 2200, f'${m:,.0f}', ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('../visuals/pay-comparison.png', dpi=150)
plt.show()

**What the table says.** Three things:

**Pay — no gap.** General business sits at $105,000 against $104,000 for data and analytics, and the 95% ranges overlap — statistically, that's the same pay. Moving toward the business roles isn't a pay cut. (It also isn't the premium the uncorrected grouping suggested. The correction at the top of this notebook is the whole story there.)

**Volume — basically the same.** 13,672 against 14,035. Both are active, steady routes to sponsorship.

**STEM — the catch, and the honest version of it.** General business comes out 0% STEM and data and analytics 100%. I'll be straight about why: the STEM label and the role groups are both built from the same occupation codes, so a business role *can't* land in STEM and an analytics role *has* to. The split isn't something the data revealed — it confirms a known rule: HR, consulting, and management jobs aren't STEM-designated and computing ones are. The volume and pay comparisons are the real findings; this one is the rule showing up exactly as expected.

**What it means.** The business roles sponsor heavily and pay the same at the median — but being non-STEM, they give no 24-month STEM OPT extension, so the H-1B clock is tighter. If the longest possible runway matters, the analytics roles are the safer route, even though the business roles look just as good on everything else.

## Reflection log

The real back-and-forths at each decision point: what I almost did, why it turned out wrong or overcomplicated, and what I did instead.

**1. The duplicate rows.** *Almost did:* `drop_duplicates()`, because "clean data has no repeats." *Why wrong here:* identical rows can be real, separate filings — one batch repeats 131 times. Auto-deleting them would have wiped out the exact thing I'm measuring, and hit the biggest sponsors hardest. *Instead:* kept them. How often a company files is a real sign of willingness to sponsor, over and over, with money behind it. Upfront in the notebook: this measures filing *activity*, not headcount.

**2. The wage conversion, and the typo trap.** *Almost did:* convert every wage by pay period and trust the unit labels. *Why wrong:* a $195,870 salary marked "per hour" becomes $407 million. *The check that saved it:* I wondered whether capping wages would wrongly delete real high earners, so instead of guessing I checked the job and the level — the extreme rows are all wrong-unit typos. *Instead:* delete nothing, invent no cutoff, report the median (which the typos can't move, and which is right for lopsided pay anyway).

**3. The role filter, made simpler.** *Overcomplicated first:* a long keyword list on hand-typed titles, then hand-editing out the IT roles that words like "manager" wrongly drag in. Fiddly, fragile, never quite clean. *The simpler way:* the government's occupation codes. Two short rules replaced the whole keyword-and-hand-edit process — and the STEM designation came along for free. The simpler tool was also the more accurate one.

**4. Deciding what to chart.** *Almost did:* chart pay directly. *Why wrong:* the typo rows stretch any pay chart so far the real range becomes an unreadable sliver. *Instead:* chart counts, keep pay as the median, save the pay comparison for the grouping where I can control the range. The limit pushed me toward a cleaner, more honest set of charts than the obvious version.

**5. The employer chart caught its own mistake.** *Missed at first:* Wayfair listed twice (`WAYFAIR LLC` / `Wayfair LLC`) — one real sponsor split in two. *The fix I said no to:* cleaning every employer name. Deep hole, deadline risk, breaks tested numbers. *Instead:* left names as recorded and called it out under the chart. Naming the flaw honestly beats a rushed half-fix.

**6b. The comparison I didn't do.** *What I tried:* merging in BLS wage data to benchmark the H-1B medians against the broader Massachusetts labor market. *Why I dropped it:* the merge fought back and the groupby already answered the question on its own — chasing it would have been scope creep dressed up as rigor. *Kept as a noted next step instead of a half-done section.*

**7. The IT-manager correction (added in the rebuild).** *What I missed:* family 11/13 includes an IT occupation — 2,583 Computer and Information Systems Managers at a $168K median — which inflated the business pay figure by $10K. *The fix:* exclude by job title (the same occupation is coded three ways, so code-matching would miss 68 rows). The headline changed from "business pays more" to "business pays the same." That's the correction at the top of this notebook, stated plainly.

## Limits, and how to reproduce this

The honest-limits section lives in `../docs/limitations.md` — what the data can't say (filings aren't hires, no OPT signal, name variants, Kaggle's combined file, pandemic-era effects).

**To reproduce:** download the source from the link in `../data/README.md`, filter to Massachusetts worksites, and run every cell top to bottom. The functions in `../src/h1b_ma.py` hold all the logic; the notebook is the narrative over it. Requirements are pinned in `../requirements.txt`.

**Walkthrough:** [H-1B Massachusetts FY2020-2024 Analysis](https://www.youtube.com/watch?v=SEite--Lkh4) — a ~10-minute video walking through the code and the decisions.